# Router Pattern (라우터 패턴)

**라우터 패턴(Router Pattern)** 은 라우팅 단계에서 입력을 분류하고, 전문 에이전트로 라우팅하며,
결과를 통합하여 결합된 응답을 생성하는 멀티 에이전트 아키텍처입니다.

이 패턴은 조직의 지식이 서로 다른 **수직 영역(verticals)** 에 분산되어 있을 때 특히 효과적입니다.
각 영역은 고유한 도구와 프롬프트를 가진 전문 에이전트가 필요합니다.

이 튜토리얼에서는 GitHub, Notion, Slack 세 가지 지식 소스를 통합하는 라우터를 구축합니다.

**참고**: [LangChain 공식 문서 - Router Knowledge Base](https://docs.langchain.com/oss/python/langchain/multi-agent/router-knowledge-base)

In [ ]:
# LangSmith 추적 (선택적)

In [ ]:
# 메인 모델과 라우터용 경량 모델

## 1. 상태 정의

라우터의 상태를 정의합니다.

In [ ]:
# 각 에이전트 노드에 전달되는 입력 (Send로 전달되는 상태)
class AgentInput(TypedDict):
# 분류 결과
class Classification(TypedDict):
# 에이전트 출력
class AgentOutput(TypedDict):
# 라우터 상태 (results는 reducer로 병렬 결과 수집)
class RouterState(TypedDict):

## 2. 도구 정의

각 지식 소스별로 도구를 정의합니다.

In [ ]:
# GitHub 도구
def search_code(query: str, repo: str = "main") -> str:
def search_issues(query: str) -> str:
def search_prs(query: str) -> str:
# Notion 도구
def search_notion(query: str) -> str:
def get_page(page_id: str) -> str:
# Slack 도구
def search_slack(query: str) -> str:
def get_thread(thread_id: str) -> str:

## 3. 전문 에이전트 생성

각 지식 소스별로 전문 에이전트를 생성합니다.

In [ ]:
# GitHub 에이전트
# Notion 에이전트
# Slack 에이전트

## 4. 쿼리 분류

사용자 쿼리를 분석하여 어떤 에이전트를 호출할지 결정합니다.

In [ ]:
# 구조화된 출력 스키마
class ClassificationResult(BaseModel):
def classify_query(state: RouterState) -> dict:

## 5. 라우팅 및 에이전트 노드 (병렬 실행)

route_to_agents는 Send 목록을 반환하여 선택된 에이전트들을 병렬로 실행합니다.
각 에이전트 노드는 AgentInput을 받아 결과만 반환합니다.

In [ ]:
# 병렬 분기 생성
def route_to_agents(state: RouterState) -> list:
        # Send(노드 이름, 해당 노드에 전달할 상태)
# ------------------------------------------------------------
# GitHub 에이전트 호출 노드
# ------------------------------------------------------------
# 이 함수는 Graph의 "github" 노드에 해당합니다.
# AgentInput 형태의 state를 입력받아 GitHub 전문 에이전트를 실행합니다.
def query_github(state: AgentInput) -> dict:
    # GitHub 전문 Agent 실행
    # 결과를 RouterState.results에 추가할 형식으로 반환
# ------------------------------------------------------------
# Notion 에이전트 호출 노드
# ------------------------------------------------------------
def query_notion(state: AgentInput) -> dict:
# ------------------------------------------------------------
# Slack 에이전트 호출 노드
# ------------------------------------------------------------
def query_slack(state: AgentInput) -> dict:

## 6. 결과 통합 (synthesize)

병렬로 수집된 results를 reducer가 합쳤으므로, 이를 바탕으로 최종 답변을 생성합니다.

In [ ]:
def synthesize_answer(state: RouterState) -> dict:
    # 결과 요약
    # 최종 답변 생성

## 7. 그래프 구성 (병렬 라우팅)

classify 후 route_to_agents가 반환하는 Send 목록에 따라 github/notion/slack 노드가 병렬 실행되고,
각 결과가 reducer(operator.add)로 results에 수집된 뒤 synthesize로 넘어갑니다.

## 8. 라우터 실행

다양한 쿼리로 라우터를 테스트합니다.

In [ ]:
# 쿼리 1: API 인증

## 9. 스트리밍 실행

각 단계를 스트리밍으로 확인합니다.

## 10. 실전 예제: 다양한 쿼리 패턴

## 주요 포인트 정리

1. **라우터 패턴**: 쿼리 분류 → 전문 에이전트로 라우팅 → 결과 통합
2. **병렬 처리**: 여러 에이전트를 동시에 호출하여 효율성 향상
3. **전문 에이전트**: 각 지식 소스별로 최적화된 에이전트
4. **결과 통합**: 여러 소스의 정보를 하나의 일관된 답변으로 통합
5. **확장성**: 새로운 지식 소스를 쉽게 추가 가능

**다음 단계**: 
- [340_Skills_Pattern.py](340_Skills_Pattern.py)에서 스킬 기반 패턴 학습
- [310_Subagents_Pattern.py](310_Subagents_Pattern.py)에서 하위 에이전트 패턴 비교